# Combined Model Full-Track Waterfall

Loads the pretrained `LitS4CombinedModel` and the `LitCombinedDataModule`
directly from the training checkpoint, iterates over all test-set segments,
and produces a time vs. frequency waterfall plot coloured by scaled
signal-to-noise ratio.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, fftshift

from src.models.model import LitS4CombinedModel
from src.models.networks import S4DCombinedModel
from src.data.data import LitCombinedDataModule

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

In [ ]:
# ── Update this path ──────────────────────────────────────────────────────────
path            = '../runs/combined_denoise_regression/lightning_logs/<run_id>'
CHECKPOINT_PATH = os.path.join(path, 'checkpoints/last.ckpt')
CONFIG_PATH     = os.path.join(path, '../../config.yaml')
# ─────────────────────────────────────────────────────────────────────────────

# Baseband half-width to display around the carrier frequency [Hz]
DISPLAY_BW_HZ = 6e6   # ±6 MHz → 12 MHz total display bandwidth

# Sampling rate (must match noise_model in src/utils/noise.py)
FS = 403e6   # Hz

## 2. Load Data Module from Checkpoint

In [ ]:
dm = LitCombinedDataModule.load_from_checkpoint(CHECKPOINT_PATH)
test_loader = dm.test_dataloader()

CUTOFF = dm.hparams.cutoff
# Index of 'avg_carrier_frequency_Hz' in the observables list
carrier_idx = dm.observables.index('avg_carrier_frequency_Hz')

print(f'Test batches  : {len(test_loader)}')
print(f'Input channels: {dm.inputs}')
print(f'Targets       : {dm.variables}')
print(f'Observables   : {dm.observables}')
print(f'Cutoff        : {CUTOFF} samples  ({CUTOFF / FS * 1e6:.1f} µs per segment)')

## 3. Load Model from Checkpoint

In [ ]:
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

enc_cfg = config['model']['init_args']['encoder']['init_args']

encoder = S4DCombinedModel(
    d_input            = enc_cfg.get('d_input', 2),
    d_output           = enc_cfg.get('d_output', 2),
    denoiser_d_model   = enc_cfg.get('denoiser_d_model', 128),
    denoiser_n_layers  = enc_cfg.get('denoiser_n_layers', 6),
    denoiser_dropout   = enc_cfg.get('denoiser_dropout', 0.0),
    denoiser_prenorm   = enc_cfg.get('denoiser_prenorm', False),
    regressor_d_model  = enc_cfg.get('regressor_d_model', 128),
    regressor_n_layers = enc_cfg.get('regressor_n_layers', 6),
    regressor_dropout  = enc_cfg.get('regressor_dropout', 0.0),
    regressor_prenorm  = enc_cfg.get('regressor_prenorm', False),
    regressor_fc_hidden= enc_cfg.get('regressor_fc_hidden', [64, 32]),
)

model = LitS4CombinedModel.load_from_checkpoint(CHECKPOINT_PATH, encoder=encoder)
model = model.to(device).eval()
print('Model loaded successfully')
print(f'  Denoiser : d_model={enc_cfg.get("denoiser_d_model", 128)}, '
      f'n_layers={enc_cfg.get("denoiser_n_layers", 6)}')
print(f'  Regressor: d_model={enc_cfg.get("regressor_d_model", 128)}, '
      f'n_layers={enc_cfg.get("regressor_n_layers", 6)}')

## 4. Run Model over Test Set and Collect Spectra

The DataModule already adds noise and normalises each segment, so no
pre-processing is needed here. `forward()` returns `(x_denoised, y_pred)`;
we keep the denoised I/Q and compute its power spectrum.

In [ ]:
# Baseband frequency axis (shifted so DC is in the centre)
baseband_freqs_hz = fftshift(fftfreq(CUTOFF, d=1.0 / FS))  # Hz
freq_mask         = np.abs(baseband_freqs_hz) <= DISPLAY_BW_HZ
n_display         = int(freq_mask.sum())

spectra_denoised = []   # denoised power per segment
spectra_noisy    = []   # noisy power per segment (for comparison)
carrier_freqs_hz = []   # per-segment carrier frequency from observables

for x_noisy, x_clean, y, obs in test_loader:
    with torch.no_grad():
        x_denoised, _y_pred = model(x_noisy.to(device))
    x_denoised = x_denoised.cpu().numpy()   # (B, L, 2)
    x_noisy_np = x_noisy.numpy()            # (B, L, 2)

    for i in range(x_noisy.shape[0]):
        # Denoised spectrum
        cplx_den = x_denoised[i, :, 0] + 1j * x_denoised[i, :, 1]
        spec_den = fftshift(np.abs(fft(cplx_den)) ** 2)
        spectra_denoised.append(spec_den[freq_mask])

        # Noisy spectrum
        cplx_noi = x_noisy_np[i, :, 0] + 1j * x_noisy_np[i, :, 1]
        spec_noi = fftshift(np.abs(fft(cplx_noi)) ** 2)
        spectra_noisy.append(spec_noi[freq_mask])

        carrier_freqs_hz.append(obs[i, carrier_idx].item())

# Stack into 2-D arrays: shape (n_display_freqs, n_segments)
power_den = np.stack(spectra_denoised, axis=1).astype(np.float32)  # (n_display, N)
power_noi = np.stack(spectra_noisy,    axis=1).astype(np.float32)
carrier_freqs_hz = np.array(carrier_freqs_hz)

n_segments = power_den.shape[1]
print(f'Total segments  : {n_segments}')
print(f'Power map shape : {power_den.shape}')
print(f'Carrier freq range: {carrier_freqs_hz.min()/1e9:.4f} – '
      f'{carrier_freqs_hz.max()/1e9:.4f} GHz')

## 5. Scale to SNR

Normalise each segment column by its median power (noise floor estimate),
then clip to [0, 1] using the 99.9th percentile as the upper reference.

In [ ]:
def scale_to_snr(power_map):
    noise_floor = np.median(power_map, axis=0, keepdims=True)
    noise_floor = np.where(noise_floor > 0, noise_floor, 1.0)
    snr = power_map / noise_floor
    return np.clip(snr / np.percentile(snr, 99.9), 0.0, 1.0)

snr_den = scale_to_snr(power_den)
snr_noi = scale_to_snr(power_noi)

# Frequency axis: use median carrier as the reference
ref_carrier_ghz    = np.median(carrier_freqs_hz) / 1e9
physical_freqs_ghz = ref_carrier_ghz + baseband_freqs_hz[freq_mask] / 1e9

# Time axis: treat segments as consecutive in time
seg_dt_s    = CUTOFF / FS
seg_times_s = np.arange(n_segments) * seg_dt_s + seg_dt_s / 2

# Mesh edges for pcolormesh
t_edges = np.append(seg_times_s - seg_dt_s / 2,
                    seg_times_s[-1] + seg_dt_s / 2)
df_ghz  = np.abs(physical_freqs_ghz[1] - physical_freqs_ghz[0])
f_edges = np.append(physical_freqs_ghz - df_ghz / 2,
                    physical_freqs_ghz[-1] + df_ghz / 2)

print(f'Reference carrier : {ref_carrier_ghz:.6f} GHz')
print(f'Frequency range   : {physical_freqs_ghz.min():.4f} – '
      f'{physical_freqs_ghz.max():.4f} GHz')
print(f'Time range        : {t_edges[0]:.4f} – {t_edges[-1]:.4f} s')

## 6. Waterfall Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

mesh = ax.pcolormesh(
    t_edges,
    f_edges,
    snr_den,
    cmap='Blues',
    vmin=0.0,
    vmax=1.0,
    shading='flat',
    rasterized=True,
)

cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=11)
cbar.set_ticks(np.linspace(0.1, 1.0, 10))

ax.set_xlabel('Time [s]', fontsize=12)
ax.set_ylabel('Frequency [GHz]', fontsize=12)
ax.set_title('Project 8 – Denoised test set waterfall', fontsize=13)

fig.tight_layout()
plt.show()

## 7. Side-by-side Comparison: Noisy vs. Denoised

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, data, title in [
    (axes[0], snr_noi, 'Noisy input'),
    (axes[1], snr_den, 'Denoised (combined model)'),
]:
    mesh = ax.pcolormesh(
        t_edges, f_edges, data,
        cmap='Blues', vmin=0.0, vmax=1.0,
        shading='flat', rasterized=True,
    )
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=10)
    cbar.set_ticks(np.linspace(0.1, 1.0, 10))
    ax.set_xlabel('Time [s]', fontsize=12)
    ax.set_ylabel('Frequency [GHz]', fontsize=12)
    ax.set_title(f'Project 8 – {title}', fontsize=12)

fig.tight_layout()
plt.show()